# `--replay-chat` — feed chat lines into the app without the game

**Checks:** the test flag `--replay-chat <file>` plays the lines of a JSON Lines file into the app as if the sniffer had captured
them (`crates/core/testdata/replay-sample.jsonl`: 8 Japanese lines over about 10 s, five channels). Needs an exe built with
`--features test-env` (see the README). The notebook starts **one fresh copy** with its own data folder, status file and **log file**,
reads what the app says about the replay from the log, and asks you what the window showed.

**Why this exists:** the first hand run looked wrong. This notebook says *where* it goes wrong: not started (log), started but not
finished (log), or fine in the log but not on screen (your answers).

**Before you start:** close the installed app and any other copy. Run Jupyter as Administrator. Keep the app window in view while the
replay runs (about 15 s) -- the notebook starts it for you.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))
from runbook import bridge, common, mockfeed, updater, replay
from runbook.common import Recorder, ask_text
import json, shutil, textwrap, time
rec = Recorder("replay-chat")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")
RUNS = common.RUNS / "replay"


## 0 · The exe and the sample file

In [ ]:
EXE = Path(ask_text("LOCAL_EXE", "Full path of the test-env exe (e.g. C:/.../target/release/resonance-stream.exe), then Enter: "))
if not EXE.is_file():
    raise SystemExit(f"{EXE} is not a file")
smoke = RUNS / "smoke"
shutil.rmtree(smoke, ignore_errors=True)
smoke.mkdir(parents=True)
code_, out = mockfeed.run_print_env(EXE, smoke / "data", smoke / "status.json", timeout=30)
st = mockfeed.read_status(smoke / "status.json")
flags = st.get("flags", []) if st else []
ok = code_ == 0 and "print-env" in flags
rec.auto("R0", "the exe understands the test flags", ok, f"version {st['version']}" if ok else " ".join(out.split())[-200:])
if not ok:
    raise SystemExit("This exe does not take the test flags -- build it with --features test-env.")
SAMPLE = root.parent / "crates" / "core" / "testdata" / "replay-sample.jsonl"
if not SAMPLE.is_file():
    SAMPLE = Path(ask_text("SAMPLE", "Full path of replay-sample.jsonl, then Enter: "))
ENTRIES = replay.parse_sample(SAMPLE.read_text(encoding="utf-8"))
EXPECT = replay.total_seconds(ENTRIES)
rec.auto("R0-sample", "the sample file is readable", len(ENTRIES) > 0, f"{len(ENTRIES)} lines, about {EXPECT:.0f} s, {SAMPLE}")


In [ ]:
# The bridge: a local MQTT broker the app publishes to (what it does) and listens to (commands). Needs Node 20+.
# Without it the notebook still works from the log and your answers.
BRIDGE = None
try:
    bridge.ensure_installed()
    BRIDGE = bridge.Serve(RUNS / "bridge-events.jsonl")
    rec.auto("R0-bridge", "the bridge broker started", True, BRIDGE.url)
except Exception as e:
    rec.record("R0-bridge", "the bridge broker started", "skip", f"{e} -- the replay is checked from the log only")

## 1 · Start the replay
A fresh copy with `--replay-chat`, no capture, no translator, no update check, no popups, a data folder of its own. The window opens and
the lines should start about 2 s after it is ready.

In [ ]:
exe = updater.fresh_copy(EXE, RUNS, "replay")
status, log = exe.parent / "status.json", exe.parent / "app.log"
args = mockfeed.flag_args(exe.parent / "data", status, log_file=log,
                          extra=("--no-update-check", "--replay-chat", str(SAMPLE),
                                *(("--bridge-url", BRIDGE.url) if BRIDGE else ())))
print("args:", " ".join(args))
pid = mockfeed.start_app(exe, args)
t0 = time.monotonic()
st = mockfeed.wait_status(status, lambda s: s.get("ready"), 120)
rec.auto("R1", "the copy started with the test flags and became ready", st is not None,
         f"pid {st['pid']}" if st else "the status file never said ready")
if st is None:
    print(mockfeed.log_tail(log, 25))
    mockfeed.stop_app(pid)
    raise SystemExit("The app never became ready (see its log above).")
print(f"ready after {time.monotonic() - t0:.1f} s -- watch the window now: the lines come over about {EXPECT:.0f} s")


## 2 · What the app's own log says
The notebook waits for `Replay finished` (the expected time plus 30 s). A missing line here means the replay itself failed, whatever the
window shows.

In [ ]:
deadline = time.monotonic() + EXPECT + 30
rep = replay.log_report("", len(ENTRIES))
while time.monotonic() < deadline:
    try:
        rep = replay.log_report(log.read_text(encoding="utf-8", errors="replace"), len(ENTRIES))
    except OSError:
        pass
    if rep["finished"] or rep["problems"]:
        break
    time.sleep(1)
took = time.monotonic() - t0
rec.auto("R2-start", f"the log says 'Replaying {len(ENTRIES)} chat lines'", rep["count_ok"],
         f"started={rep['started']}" if rep["started"] is not None else "no 'Replaying N chat lines' line in the log")
rec.auto("R2-end", "the log says 'Replay finished'", rep["finished"], f"{took:.0f} s after the start (expected about {EXPECT:.0f} s)")
rec.auto("R2-quiet", "no [Replay] warning or error in the log", not rep["problems"], "; ".join(rep["problems"])[:300])
if not (rep["count_ok"] and rep["finished"]) or rep["problems"]:
    print("\nreplay lines in the log:\n" + textwrap.indent("\n".join(rep["replay_lines"]) or "(none)", "  "))
    print("\nlog tail:\n" + textwrap.indent(mockfeed.log_tail(log, 25), "  "))


## 2b · What the app published on the bridge
The broker recorded every event the app sent to the window. These checks are on **what was sent**, not on what the window drew (section 3
asks about that): the right number of lines, each with its channel, sender, level and text, in order, at the pace of the file, then
`Replay finished`. B0 sends the app a command and waits for its answer.

In [ ]:
if BRIDGE is None:
    print("no bridge in this run: skipping B0-B5")
else:
    try:
        ack = BRIDGE.send("ping", timeout=10)
        rec.auto("B0", "the app answers a command sent over the bridge", ack.get("ok") is True, f"ack {ack.get('id', '')[:8]}")
    except Exception as e:
        rec.auto("B0", "the app answers a command sent over the bridge", False, str(e))
    time.sleep(1)
    result = bridge.verify("replay-chat", BRIDGE.out, SAMPLE)
    for c in result["checks"]:
        rec.auto(c["id"], c["title"], c["ok"], c["detail"])
    print("recording:", BRIDGE.out)

## 3 · What the window showed
The app is still open. Look at it (scroll if needed) and answer. Say what you saw in a note when something is off, e.g.
`fail: only WORLD has lines`.

In [ ]:
by = replay.by_channel(ENTRIES)
rec.manual("R3-1", "every line appeared, in the right tab",
           "Click each tab and compare with the sample:\n" + "\n".join(f"  {ch}: {', '.join(t[:18] for t in texts)}" for ch, texts in by.items()),
           "WORLD 4 lines (they are hidden from senders below Settings > minimum level, default 1; the sample says level 60), GUILD 1, PARTY 1, LOCAL 1, BEGINNER 1; sender names ミナト / Kenji / さくら / ゆう")
rec.manual("R3-2", "the lines arrived one after another, not all at once",
           "You were told to watch while the lines came in. If you missed it, close the app, run from section 1 again and watch.",
           "about one new line every 1-2 s over about 10 s, starting about 2 s after the window was ready")
rec.manual("R3-3", "emotes show as text placeholders",
           "PARTY line `回復お願いします！<sprite=3>` and WORLD line `emojiPic=9`.",
           "[이모지] in the PARTY line, [스티커] for the sticker line")
rec.manual("R3-4", "the system log (the log button / tab) lists the replay",
           "Open the system log in the app.",
           "'Replaying 8 chat lines from ...' and, at the end, 'Replay finished'")
rec.manual("R3-5", "unread badges and the list behave like real chat",
           "Lines in a tab you are not looking at raise its unread badge; the open tab scrolls to the newest line.",
           "badges on the other tabs while the replay runs; no duplicate rows")


## 4 · Close the app

In [ ]:
mockfeed.stop_copies(RUNS)
mockfeed.stop_app(pid)
if BRIDGE:
    BRIDGE.stop()
print("closed; files for the report are in", exe.parent)


## Report
Paste the output of the next cell to Claude, plus anything odd you saw (and the `app.log` if a row failed).

In [ ]:
print(rec.report())